# Controller 后训练 Runbook

这个 notebook 用来在 Jupyter 里跑当前主线：`prepare_round -> SFT -> GRPO -> predict_holdout -> evaluate -> annotate_queue -> next_round`。

默认配置读取 `src/task_router_graph_train/configs/controller_grpo_online.yaml`。SFT/GRPO 是长任务，默认用配置开关控制。

如果你是第一次跑，建议按这个顺序理解：

1. 先执行前两个代码单元，只是找仓库、读配置，不会训练。
2. 执行 `Prepare Round`，它只生成训练资产，通常很快。
3. 确认资产没问题后，再把配置里的 `run.sft` 改成 `true` 跑 SFT。
4. GRPO 默认是 `export_only`，只导出 verl 训练请求，不会真正启动强化学习训练。
5. `predict_holdout` 会用本地 SGLang 服务生成 holdout predictions；服务没准备好时保持关闭。
6. `evaluate` 和 `annotate_queue` 依赖前面产物，没准备好时保持关闭。

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import yaml

# Jupyter 的工作目录可能是仓库根目录，也可能是 scripts/post_training。
# 这里向上查找 src/task_router_graph_train，用它判断仓库根目录。
def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "task_router_graph_train").exists():
            return candidate
    fallback = Path("/root/WORK/task-rounting").resolve()
    if (fallback / "src" / "task_router_graph_train").exists():
        return fallback
    raise FileNotFoundError("找不到仓库根目录；请从 repo 内启动 Jupyter")


REPO_ROOT = find_repo_root(Path.cwd())

# 这个仓库没有安装成 Python package；把 src 加进 sys.path 后，notebook 才能 import 训练模块。
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("REPO_ROOT =", REPO_ROOT)
print("PYTHONPATH has src =", str(SRC_ROOT) in sys.path)

In [ ]:
# ===== 加载统一训练配置 =====
# 默认读取 controller_grpo_online.yaml。
# 如果你要临时试另一份配置，可以在启动 Jupyter 前设置：
#   export POST_TRAINING_RUN_CONFIG=/path/to/controller_grpo_online.yaml
CONFIG_PATH = Path(
    os.environ.get(
        "POST_TRAINING_RUN_CONFIG",
        REPO_ROOT / "src" / "task_router_graph_train" / "configs" / "controller_grpo_online.yaml",
    )
).resolve()
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}


# 配置里的输出路径允许写 {round_id}，例如 var/runs/.../{round_id}。
# 这里会把相对路径统一转成绝对路径，避免 notebook 工作目录变化导致写错位置。
def resolve_repo_path(raw_path: str) -> Path:
    path = Path(str(raw_path).format(round_id=ROUND_ID))
    return path if path.is_absolute() else (REPO_ROOT / path).resolve()


# round 是一次后训练迭代的名字。round_0001 会对应一组资产目录和 manifest。
ROUND_ID = str(CONFIG.get("round", {}).get("id", "round_0001")).strip()
PREVIOUS_ROUND_ID = str(CONFIG.get("round", {}).get("previous_round_id", "")).strip()

model_cfg = CONFIG.get("model", {})
update_cfg = CONFIG.get("update", {})
data_cfg = CONFIG.get("data", {})
rollout_cfg = CONFIG.get("rollout", {})
sft_cfg = CONFIG.get("sft", {})
holdout_inference_cfg = CONFIG.get("holdout_inference", {})

# BASE_MODEL 是底座模型路径。优先读环境变量，避免把本机私有模型路径写进 git。
# 示例：export BASE_MODEL=/models/Qwen2.5-3B-Instruct
BASE_MODEL = os.environ.get("BASE_MODEL", str(model_cfg.get("path", "")).strip())
LORA_TARGET_MODULES = list(model_cfg.get("target_modules", ["q_proj", "v_proj"]))

# 这些目录都是本轮训练/评测的输出位置，默认写到 var/runs 下。
outputs_cfg = CONFIG.get("outputs", {})
SFT_OUTPUT_DIR = resolve_repo_path(outputs_cfg.get("sft_dir", "var/runs/task_router_graph_train/sft/{round_id}"))
SFT_MERGED_OUTPUT_DIR = resolve_repo_path(
    outputs_cfg.get("sft_merged_dir", "var/runs/task_router_graph_train/sft/{round_id}/merged")
)
GRPO_OUTPUT_DIR = resolve_repo_path(outputs_cfg.get("grpo_dir", "var/runs/task_router_graph_train/grpo/{round_id}"))
EVAL_OUTPUT_DIR = resolve_repo_path(outputs_cfg.get("eval_dir", "var/runs/task_router_graph_train/eval/{round_id}"))
HOLDOUT_PREDICTIONS = resolve_repo_path(
    outputs_cfg.get("holdout_predictions", "var/runs/task_router_graph_train/predictions/{round_id}_holdout_predictions.jsonl")
)

# run 段是安全开关：默认只做轻量步骤，长任务必须显式打开。
run_cfg = CONFIG.get("run", {})
RUN_SFT = bool(run_cfg.get("sft", False))
RUN_GRPO_EXPORT_ONLY = bool(run_cfg.get("grpo_export_only", True))
RUN_GRPO_FULL_UPDATE = bool(run_cfg.get("grpo_full_update", False))
RUN_PREDICT_HOLDOUT = bool(run_cfg.get("predict_holdout", False))
RUN_EVALUATE = bool(run_cfg.get("evaluate", False))
RUN_ANNOTATE_QUEUE = bool(run_cfg.get("annotate_queue", False))

# SFT 复用统一配置里的 update/data/model 参数，只额外从 sft.total_epochs 读取 SFT 轮数。
SFT_PARAMS = {
    "num_train_epochs": int(sft_cfg.get("total_epochs", 5)),
    "per_device_train_batch_size": int(update_cfg.get("per_device_train_batch_size", 1)),
    "gradient_accumulation_steps": int(update_cfg.get("gradient_accumulation_steps", 4)),
    "learning_rate": float(update_cfg.get("learning_rate", 2e-4)),
    "max_seq_length": int(sft_cfg.get("max_seq_length", data_cfg.get("max_prompt_length", 2048))),
    "export_merged_model": bool(sft_cfg.get("export_merged_model", True)),
    "merged_output_dir": SFT_MERGED_OUTPUT_DIR,
    "lora_r": int(sft_cfg.get("lora_rank", 8)),
    "lora_alpha": int(sft_cfg.get("lora_alpha", 16)),
    "lora_dropout": float(sft_cfg.get("lora_dropout", model_cfg.get("lora_dropout", 0.05))),
    "seed": int(CONFIG.get("seed", 42)),
    "bf16": bool(sft_cfg.get("bf16", False)),
    "fp16": bool(sft_cfg.get("fp16", False)),
    "gradient_checkpointing": bool(sft_cfg.get("gradient_checkpointing", False)),
    "torch_empty_cache_steps": sft_cfg.get("torch_empty_cache_steps"),
    "nproc_per_node": int(sft_cfg.get("nproc_per_node", 1)),
    "nnodes": int(sft_cfg.get("nnodes", 1)),
    "node_rank": int(sft_cfg.get("node_rank", 0)),
    "master_addr": str(sft_cfg.get("master_addr", "127.0.0.1")),
    "master_port": int(sft_cfg.get("master_port", 29500)),
}
# GRPO 直接使用同一份 controller_grpo_online.yaml，不再维护第二份 grpo 配置。
GRPO_CONFIG_PATH = CONFIG_PATH
GRPO_NUM_CANDIDATES = int(rollout_cfg.get("num_candidates", 4))
GRPO_SEED = int(CONFIG.get("seed", 42))
GRPO_TRAIN_PARAMS = {
    "num_train_epochs": int(update_cfg.get("total_epochs", 1)),
    "per_device_train_batch_size": int(update_cfg.get("per_device_train_batch_size", 1)),
    "gradient_accumulation_steps": int(update_cfg.get("gradient_accumulation_steps", 4)),
    "learning_rate": float(update_cfg.get("learning_rate", 2e-4)),
    "max_seq_length": int(data_cfg.get("max_prompt_length", 2048)),
    "lora_r": int(model_cfg.get("lora_rank", 0)),
    "lora_alpha": int(model_cfg.get("lora_alpha", 0)),
    "lora_dropout": float(model_cfg.get("lora_dropout", 0.05)),
}


def resolve_grpo_model_path() -> str:
    # GRPO/SGLang direct update 当前走 full-rank 权重同步，不能直接消费 SFT LoRA adapter。
    # 优先使用 SFT 合并后的 full model；不存在时才退回 BASE_MODEL。
    if (SFT_MERGED_OUTPUT_DIR / "config.json").exists():
        return str(SFT_MERGED_OUTPUT_DIR)
    return str(BASE_MODEL)

HOLDOUT_INFERENCE_CONFIG = {
    "base_url": str(holdout_inference_cfg.get("base_url", "http://127.0.0.1:30000/v1")),
    "api_key_env": str(holdout_inference_cfg.get("api_key_env", "SGLANG_API_KEY")),
    "model": str(holdout_inference_cfg.get("model", "qwen35-4b")),
    "timeout_sec": float(holdout_inference_cfg.get("timeout_sec", 60)),
    "max_tokens": int(holdout_inference_cfg.get("max_tokens", 512)),
    "temperature": float(holdout_inference_cfg.get("temperature", 0.0)),
    "max_samples": holdout_inference_cfg.get("max_samples"),
}

# 打印最终解析结果。跑长任务前先看这里，确认 round、模型、输出目录和开关都符合预期。
print(json.dumps({
    "CONFIG_PATH": str(CONFIG_PATH),
    "ROUND_ID": ROUND_ID,
    "PREVIOUS_ROUND_ID": PREVIOUS_ROUND_ID,
    "BASE_MODEL": BASE_MODEL,
    "SFT_OUTPUT_DIR": str(SFT_OUTPUT_DIR),
    "SFT_MERGED_OUTPUT_DIR": str(SFT_MERGED_OUTPUT_DIR),
    "GRPO_MODEL_PATH_CANDIDATE": resolve_grpo_model_path(),
    "GRPO_OUTPUT_DIR": str(GRPO_OUTPUT_DIR),
    "EVAL_OUTPUT_DIR": str(EVAL_OUTPUT_DIR),
    "HOLDOUT_PREDICTIONS": str(HOLDOUT_PREDICTIONS),
    "RUN_SFT": RUN_SFT,
    "SFT_NPROC_PER_NODE": SFT_PARAMS["nproc_per_node"],
    "RUN_GRPO_EXPORT_ONLY": RUN_GRPO_EXPORT_ONLY,
    "RUN_GRPO_FULL_UPDATE": RUN_GRPO_FULL_UPDATE,
    "RUN_PREDICT_HOLDOUT": RUN_PREDICT_HOLDOUT,
    "HOLDOUT_INFERENCE_BASE_URL": HOLDOUT_INFERENCE_CONFIG["base_url"],
    "HOLDOUT_INFERENCE_MODEL": HOLDOUT_INFERENCE_CONFIG["model"],
    "GRPO_N_GPUS_PER_NODE": int(update_cfg.get("n_gpus_per_node", 1)),
    "GRPO_TENSOR_MODEL_PARALLEL_SIZE": int(rollout_cfg.get("tensor_model_parallel_size", 1)),
    "GRPO_DATA_PARALLEL_SIZE": int(rollout_cfg.get("data_parallel_size", 1)),
    "GRPO_CONFIG_PATH": str(GRPO_CONFIG_PATH),
}, ensure_ascii=False, indent=2))

## 1. Prepare Round

生成当前 round 的 SFT、GRPO、holdout、teacher queue 等资产。

你可以把它理解成“备料”：它不会训练模型，只是把 frozen 的 `manual_protocol_v1` 和上一轮接纳的 `sft_admissions` 整理成这一轮要用的文件。

执行后重点看输出里的 `round_manifest_path` 和 `counts_by_split`。如果这里失败，后面的 SFT/GRPO 都不要继续跑。

In [ ]:
from task_router_graph_train.dataset import prepare_round_assets

# 生成当前 round 的资产目录。previous_round_id 为空时，只使用 manual_protocol_v1 基础数据。
prepare_report = prepare_round_assets(
    round_id=ROUND_ID,
    previous_round_id=PREVIOUS_ROUND_ID.strip() or None,
    workspace_root=REPO_ROOT,
)

print(json.dumps(prepare_report, ensure_ascii=False, indent=2))

## 2. SFT

把配置文件里的 `run.sft` 改成 `true` 后执行。第一次建议小 epoch 跑通链路，再加训练量。

SFT 是监督微调：输入是 controller 看到的 state，目标输出是一个 JSON action。它主要让模型先学会格式、动作空间和 environment 事实读取。

单机多卡时，直接在统一配置里调整 `sft.nproc_per_node`；notebook 会自动走 `torchrun` 拉起多进程，不需要手写分布式命令。

跑之前必须确认 `BASE_MODEL` 有值；否则代码不知道从哪个底座模型开始训练。

In [ ]:
from task_router_graph_train.train import train_controller_sft

# RUN_SFT=false 时，这个单元只打印跳过信息，方便你安全地从上到下执行 notebook。
if RUN_SFT:
    if not BASE_MODEL:
        raise ValueError("请先在配置单元设置 BASE_MODEL，或设置环境变量 BASE_MODEL")

    # 训练输出会写到 SFT_OUTPUT_DIR，通常是 LoRA adapter、tokenizer 和 metrics。
    sft_report = train_controller_sft(
        round_id=ROUND_ID,
        model_name_or_path=BASE_MODEL,
        lora_target_modules=LORA_TARGET_MODULES,
        output_dir=SFT_OUTPUT_DIR,
        **SFT_PARAMS,
    )
    print(json.dumps(sft_report, ensure_ascii=False, indent=2))
else:
    print("跳过 SFT。需要训练时把配置 run.sft 改成 true。")

## 3. GRPO

`run.grpo_export_only=true` 只导出 verl 数据和请求文件，不启动训练。真正训练需要安装 `verl`，并设置 teacher API key，例如 `os.environ["API_KEY_Qwen"] = "..."`。

GRPO 是在 SFT 之后继续优化 controller policy。这里会用同一份配置里的 rollout/update/teacher 设置：rollout 负责采样候选动作，teacher 负责给候选动作排序，verl 负责更新模型。

单机多卡时，重点检查 `update.n_gpus_per_node`、`rollout.tensor_model_parallel_size` 和 `rollout.data_parallel_size` 是否匹配；GRPO 失败后会尝试清理本次 verl/Ray 进程组，避免残留继续占卡。

新手建议先保持 `run.grpo_export_only=true`、`run.grpo_full_update=false`，确认导出的 `verl_training_request.json` 没问题后再打开 full update。

In [ ]:
import importlib
from task_router_graph_train.train import controller_grpo as controller_grpo_module

# Jupyter 会缓存已导入模块；这里强制 reload，避免长任务继续跑旧版 GRPO 启动逻辑。
controller_grpo_module = importlib.reload(controller_grpo_module)
train_controller_grpo = controller_grpo_module.train_controller_grpo

# export_only 模式只导出数据和 verl 请求；full_update 才会真正调用 verl 开始训练。
if RUN_GRPO_EXPORT_ONLY or RUN_GRPO_FULL_UPDATE:
    grpo_model_path = resolve_grpo_model_path()
    if not grpo_model_path:
        raise ValueError("请先提供 BASE_MODEL，或先导出 SFT merged model")
    print("GRPO model =", grpo_model_path)

    # config_path 使用当前统一配置文件，确保 teacher、rollout、update 读的是同一份参数。
    grpo_report = train_controller_grpo(
        round_id=ROUND_ID,
        config_path=GRPO_CONFIG_PATH,
        output_dir=GRPO_OUTPUT_DIR,
        runtime_root=REPO_ROOT,
        model_name_or_path=grpo_model_path,
        lora_target_modules=LORA_TARGET_MODULES,
        num_candidates=GRPO_NUM_CANDIDATES,
        seed=GRPO_SEED,
        **GRPO_TRAIN_PARAMS,
        export_only=not RUN_GRPO_FULL_UPDATE,
    )
    print(json.dumps(grpo_report, ensure_ascii=False, indent=2))
else:
    print("跳过 GRPO。需要导出数据时把配置 run.grpo_export_only 改成 true；需要训练时把 run.grpo_full_update 改成 true。")

## 4. Holdout Predict

用当前模型对 round 的 `holdout_records.jsonl` 生成配置里的 `outputs.holdout_predictions`。默认走本地 SGLang 的 OpenAI-compatible `/v1/chat/completions` 接口。

打开 `run.predict_holdout=true` 后，这个单元会直接写出评测需要的 predictions.jsonl；如果本地 SGLang 没启动或 API key 环境变量没设置，会在这里直接失败。

In [ ]:
from task_router_graph_train.eval import generate_holdout_predictions
from task_router_graph_train.rounds import load_round_manifest, resolve_round_asset_path

# holdout_records.jsonl 是当前 round 的评测样本来源。predictions 则是本 notebook 会写出的预测文件。
manifest = load_round_manifest(round_id=ROUND_ID)
holdout_records = resolve_round_asset_path(manifest, "holdout_records")
print("holdout_records =", holdout_records)
print("predictions =", HOLDOUT_PREDICTIONS)
print(json.dumps(HOLDOUT_INFERENCE_CONFIG, ensure_ascii=False, indent=2))

# RUN_PREDICT_HOLDOUT=true 时才真正请求 SGLang；否则只打印路径和配置。
if RUN_PREDICT_HOLDOUT:
    holdout_predict_report = generate_holdout_predictions(
        record_path=holdout_records,
        output_path=HOLDOUT_PREDICTIONS,
        **HOLDOUT_INFERENCE_CONFIG,
    )
    print(json.dumps(holdout_predict_report, ensure_ascii=False, indent=2))
else:
    print("跳过 holdout predict。需要自动生成 predictions 时把配置 run.predict_holdout 改成 true。")

## 5. Holdout Evaluate

上一单元会把 holdout predictions 写到配置里的 `outputs.holdout_predictions`。这个单元读取同一份文件做评测，失败样本可以自动进入当前 round 的 `teacher_queue.jsonl`。

如果 `run.evaluate=false`，这个单元只会打印 holdout 和 predictions 路径，方便你确认评测输入输出是否对齐。

In [ ]:
from task_router_graph_train.eval import evaluate_holdout_predictions
from task_router_graph_train.rounds import load_round_manifest, resolve_round_asset_path

# 从 round manifest 找到 holdout_records.jsonl。manifest 是当前 round 的资产索引。
manifest = load_round_manifest(round_id=ROUND_ID)
holdout_records = resolve_round_asset_path(manifest, "holdout_records")
print("holdout_records =", holdout_records)
print("predictions =", HOLDOUT_PREDICTIONS)

# RUN_EVALUATE=true 时才真正评测；否则只打印路径。
if RUN_EVALUATE:
    if not HOLDOUT_PREDICTIONS.exists():
        raise FileNotFoundError(f"缺少 predictions.jsonl: {HOLDOUT_PREDICTIONS}")

    prediction_rows = [json.loads(line) for line in HOLDOUT_PREDICTIONS.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not prediction_rows:
        raise ValueError(f"predictions.jsonl 为空: {HOLDOUT_PREDICTIONS}")
    if any(not str(row.get("sample_id", "")).strip() for row in prediction_rows if isinstance(row, dict)):
        raise ValueError(f"predictions.jsonl 存在缺少 sample_id 的行: {HOLDOUT_PREDICTIONS}")

    # enqueue_failed_badcases=True 会把未通过的 holdout 样本放入 teacher_queue，供后续标注回流。
    eval_report = evaluate_holdout_predictions(
        record_path=holdout_records,
        prediction_path=HOLDOUT_PREDICTIONS,
        enqueue_failed_badcases=True,
        badcase_round_manifest=Path(manifest["_manifest_path"]),
    )
    EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (EVAL_OUTPUT_DIR / "metrics_summary.json").write_text(
        json.dumps(eval_report["metrics_summary"], ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    (EVAL_OUTPUT_DIR / "evidence_rows.jsonl").write_text(
        "\n".join(json.dumps(row, ensure_ascii=False) for row in eval_report["evidence_rows"]) + "\n",
        encoding="utf-8",
    )
    print(json.dumps(eval_report["metrics_summary"], ensure_ascii=False, indent=2))
else:
    print("跳过 evaluate。需要评测时把配置 run.evaluate 改成 true。")

## 6. Evaluation Chart

把 `metrics_summary` 直接渲染成 notebook 内可见的 HTML/SVG 柱状图，不依赖 matplotlib。

In [ ]:
from IPython.display import HTML, display
from task_router_graph_train.eval import render_metrics_summary_chart_html

metrics_summary_path = EVAL_OUTPUT_DIR / "metrics_summary.json"
chart_metrics = None
if "eval_report" in globals() and isinstance(eval_report, dict):
    chart_metrics = eval_report.get("metrics_summary", {})
elif metrics_summary_path.exists():
    chart_metrics = json.loads(metrics_summary_path.read_text(encoding="utf-8"))

if chart_metrics:
    display(HTML(render_metrics_summary_chart_html(chart_metrics)))
    print(json.dumps(chart_metrics, ensure_ascii=False, indent=2))
else:
    print("跳过图表。先执行 Holdout Evaluate，或确认 metrics_summary.json 已生成。")

## 7. Annotate Queue

把 `teacher_queue.jsonl` 里的 badcase 交给 teacher，接纳的样本写入 `sft_admissions.jsonl`。下一轮 `prepare_round` 用 `previous_round_id` 合入。

这一步会调用在线 teacher，所以需要配置里的 `teacher.*` 可用，并且 `API_KEY_Qwen` 等环境变量已经设置。没有 badcase 或没有 key 时，不要打开这个开关。

In [ ]:
from task_router_graph_train.feedback import annotate_teacher_queue

# RUN_ANNOTATE_QUEUE=true 时才会请求 teacher 标注；默认关闭，避免误调用外部模型接口。
if RUN_ANNOTATE_QUEUE:
    annotation_report = annotate_teacher_queue(round_id=ROUND_ID)
    print(json.dumps(annotation_report, ensure_ascii=False, indent=2))
else:
    print("跳过 annotate_queue。需要回流 SFT 样本时把配置 run.annotate_queue 改成 true。")

## 8. 下一轮

下一轮把统一配置里的 round 改成：

```yaml
round:
  id: round_0002
  previous_round_id: round_0001
```

然后从 `Prepare Round` 重新执行。

`previous_round_id` 的作用是读取上一轮的 `sft_admissions.jsonl`，把 teacher 接纳的 badcase 加进下一轮 SFT 数据里。